# DR retraining (APTOS 2019) - teachers, ConvNeXt student, equal-weight / augmentation ablations, Grad-CAM
Runtime > Change runtime type > **GPU**. The run retrains the APTOS teachers and student; the student ConvNeXt and teacher backbones use ImageNet initialization. Validation QWK selects checkpoints; the test split is held out from that selection. If Colab disconnects, re-run setup then the stage cell; checkpoints synchronize to Drive and resume with saved random states.

Drive layout expected (see README.md):
- `MyDrive/DR_Research/colab_retrain.zip`  (zip of the `colab_retrain` folder)  **or** push the folder to GitHub and set `USE_GIT=True`
- `MyDrive/DR_Research/data/train.csv` and `MyDrive/DR_Research/data/cache_224.npz`

In [ ]:
# 1) Setup: Drive, code, packages, GPU
import os, sys, subprocess, zipfile, shutil
os.chdir('/content')
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/DR_Research'
RUN_ID = 'aptos_fullfactorial_20261011_v2'  # New teacher-objective checkpoint layout; keep this ID to resume this run.
USE_GIT = False   # True: clone from GitHub (folder colab_retrain must be pushed first)
WORK = '/content/colab_retrain'

if os.path.isdir(WORK):
    shutil.rmtree(WORK)
if USE_GIT:
    subprocess.run('git clone --depth 1 https://github.com/EceKaraalp/dr-teacher-assisted-residual-grading.git /content/repo', shell=True, check=True)
    shutil.copytree('/content/repo/colab_retrain', WORK)
elif os.path.isfile(DRIVE_ROOT + '/colab_retrain/dr_common.py'):
    shutil.copytree(DRIVE_ROOT + '/colab_retrain', WORK)
else:
    shutil.rmtree('/content/_unz', ignore_errors=True)
    with zipfile.ZipFile(DRIVE_ROOT + '/colab_retrain.zip') as z:
        print('ZIP entries:', z.namelist()[:5])
        z.extractall('/content/_unz')
    hits = [r for r, _, fs in os.walk('/content/_unz') if 'dr_common.py' in fs]
    assert hits, 'dr_common.py is not inside the ZIP; re-upload colab_retrain.zip'
    shutil.copytree(hits[0], WORK)

os.chdir(WORK); sys.path.insert(0, WORK)
assert os.path.isfile(WORK + '/dr_common.py'), 'dr_common.py not found in ' + WORK
r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm'], capture_output=True, text=True, cwd='/content')
if r.returncode != 0:
    print(r.stdout[-1500:], r.stderr[-1500:])
import importlib; importlib.invalidate_caches()

# data: copy to local disk for speed
os.makedirs('/content/data', exist_ok=True)
for f in ['train.csv', 'cache_224.npz']:
    if not os.path.exists('/content/data/' + f):
        shutil.copy(f'{DRIVE_ROOT}/data/{f}', '/content/data/' + f)

# environment for run_all.py
os.environ['DATA_DIR'] = '/content/data'
LOCAL_RESULTS = '/content/' + RUN_ID
DRIVE_RESULTS = DRIVE_ROOT + '/results_retrain/' + RUN_ID
if os.path.isdir(DRIVE_RESULTS):
    shutil.copytree(DRIVE_RESULTS, LOCAL_RESULTS, dirs_exist_ok=True)  # restore checkpoints/results after a runtime reset
os.environ['OUT_DIR'] = LOCAL_RESULTS
os.environ['DRIVE_DIR'] = DRIVE_RESULTS   # results/checkpoints are mirrored here during training
os.environ['SEEDS'] = '42,52,62'
os.environ['ARMS'] = 'teacher_gate__student_gated,teacher_gate__student_equal,teacher_equalprob__student_gated,teacher_equalprob__student_equal,teacher_equallogit__student_gated,teacher_equallogit__student_equal,teacher_t1__student_gated,teacher_t1__student_equal,teacher_t2__student_gated,teacher_t2__student_equal,teacher_gate__student_gated_aug'
os.environ['BATCH'] = '16'
os.environ.pop('DR_SMOKE', None); os.environ.pop('DR_MAX_EPOCHS', None)

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - enable GPU runtime!')
assert torch.cuda.is_available(), 'No GPU'

In [ ]:
# 2) OPTIONAL smoke test (~5-10 min): tiny data, 1 epoch, checks the whole pipeline. Skip for the real run.
RUN_SMOKE = False
if RUN_SMOKE:
    env = dict(os.environ, DR_SMOKE='1', DR_SMOKE_N='600', DR_MAX_EPOCHS='1', SEEDS='42', ARMS='teacher_gate__student_gated,teacher_gate__student_equal', OUT_DIR='/content/smoke', DRIVE_DIR='')
    for st in ['teachers', 'students', 'analyze']:
        r = subprocess.run([sys.executable, '-W', 'ignore', 'run_all.py', '--stage', st], env=env)
        assert r.returncode == 0, st
    print('SMOKE OK')

In [ ]:
# 3) Data, split, and exact-manifest verification (2928 / 367 / 367)
import json, numpy as np, dr_common as C
d = C.get_data('/content/data', batch_size=16)
counts = {k: len(v) for k, v in d['ids'].items()}
assert counts == {'train': 2928, 'val': 367, 'test': 367}, counts
assert d['images'].shape == (3662, 224, 224, 3) and d['images'].dtype == np.uint8
print(counts, 'class weights:', {k: round(v, 3) for k, v in d['class_weights'].items()})
ref = json.load(open('reference/split_manifest_original.json'))
for sp in ('train', 'val', 'test'):
    same = sorted(map(str, ref['sample_ids'][sp])) == sorted(map(str, d['ids'][sp]))
    print(sp, 'matches frozen manifest:', same)
    assert same, f'{sp} differs from the frozen reference split'
groups = C.duplicate_groups(d['images'])
print('exact-duplicate groups:', len(groups))

In [ ]:
# 4) STAGE 1 - Teacher 1 baseline + three separately trained Teacher 2 fusion objectives (gate/equal probability/equal logits). Resumable.
!python -W ignore run_all.py --stage teachers --seed 42

In [ ]:
# 5) STAGE 2 - 12 student arms x 3 seeds (36 runs). Includes augmentation on the manuscript-aligned equal-logit teacher/equal-student final system. Main teacher fusion arms use the matching separately trained Teacher 2 objective checkpoint. Resumable; completed arms are skipped.
!python -W ignore run_all.py --stage students

In [ ]:
# 6) STAGE 3 - analysis: all validation/test metrics, per-sample probabilities, gate/equal identity checks,
#    paired bootstrap, teacher/augmentation ablations, duplicate audit, latency and parameter counts
!python -W ignore run_all.py --stage analyze

In [ ]:
# 7) Grad-CAM (Reviewer 2) for the final gated model, seed 42
os.environ['ARM'] = 'teacher_gate__student_gated'; os.environ['SEED'] = '42'; os.environ['N_PER_CLASS'] = '3'
!python -W ignore gradcam.py
from IPython.display import Image, display
display(Image(LOCAL_RESULTS + '/gradcam/gradcam_teacher_gate__student_gated_seed42.png'))

In [ ]:
# 8) OPTIONAL external-hospital/dataset evaluation (evaluation only; no external-data training/tuning)
# Provide independently labeled images in Drive: external/labels.csv (id_code,diagnosis) + external/images/.
# diagnosis mapping must be 0=No DR, 1=Mild, 2=Moderate, 3=Severe, 4=Proliferative.
RUN_EXTERNAL = False
EXT = DRIVE_ROOT + '/external'
if RUN_EXTERNAL:
    subprocess.run([sys.executable, 'external_eval.py', '--csv', EXT + '/labels.csv',
                    '--images', EXT + '/images', '--out', LOCAL_RESULTS + '/external',
                    '--results-root', LOCAL_RESULTS, '--arm', 'teacher_gate__student_gated'], check=True)
    C.sync_dirs(LOCAL_RESULTS, DRIVE_RESULTS)

In [ ]:
# 9) Show key results and mirror everything to Drive
import json, pandas as pd
R = LOCAL_RESULTS + '/analysis'
print(pd.read_csv(R + '/student_summary_mean_sd.csv', header=[0, 1], index_col=0).round(4).to_string())
rep = json.load(open(R + '/report.json'))
for k, v in rep.items():
    if k in ('inference_gate_vs_equal_swap', 'retrained_gate_vs_equal', 'teacher_pairwise_test',
             'augmentation_vs_no_augmentation', 'system_vs_student_only', 'latency_and_params',
             'test_images_with_development_duplicate'):
        print(k, json.dumps(v, indent=1)[:900])
C.sync_dirs(LOCAL_RESULTS, DRIVE_RESULTS)
print('mirrored to', DRIVE_RESULTS)